<a href="https://colab.research.google.com/github/Linux-Server/Transformers/blob/main/Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "EleutherAI/pythia-410m"
model = AutoModelForCausalLM.from_pretrained(model_name,
low_cpu_mem_usage=True, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [49]:
text = "Once upon a time, there was a"

encoded_text = tokenizer(text, return_tensors="pt").to("cuda")
encoded_text

{'input_ids': tensor([[10758,  2220,   247,   673,    13,   627,   369,   247]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [50]:
encoded_out = model.generate(**encoded_text)
encoded_out

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


tensor([[10758,  2220,   247,   673,    13,   627,   369,   247,   637,   665,
           369,   247,  1270,   187,   395,  6422,  6963,    15,   754,   369,
           247,  1270,   285,  6422,  6963,    13,   285,   344]],
       device='cuda:0')

In [51]:
tokenizer.decode(encoded_out[0])

'Once upon a time, there was a man who was a great\nand powerful king. He was a great and powerful king, and he'

In [52]:
def compute_model_size(model, precision_bytes=4):

  num_param  = sum(p.numel() for p in model.parameters())
  return f"The size of the model is : {(num_param * precision_bytes )/ 1024 ** 3} GB"

In [53]:
model.gpt_neox.layers[0].attention.dense.weight

Parameter containing:
tensor([[ 0.0061, -0.0016, -0.0068,  ..., -0.0062,  0.0138,  0.0222],
        [ 0.0077,  0.0157, -0.0090,  ...,  0.0013, -0.0132,  0.0109],
        [-0.0330,  0.0008,  0.0281,  ...,  0.0026,  0.0456, -0.0077],
        ...,
        [-0.0105,  0.0091, -0.0137,  ..., -0.0046,  0.0371, -0.0077],
        [-0.0063,  0.0035,  0.0147,  ...,  0.0220,  0.0158,  0.0224],
        [-0.0299,  0.0129,  0.0208,  ..., -0.0040, -0.0065,  0.0122]],
       device='cuda:0', requires_grad=True)

In [54]:
from transformers import QuantoConfig

quatization_cofig = QuantoConfig(weights="int8")

model_name = "EleutherAI/pythia-410m"
model_one = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quatization_cofig, device_map="auto")



In [55]:
model_one.gpt_neox.layers[0].attention.dense.weight

<class 'optimum.quanto.tensor.weights.qbytes.WeightQBytesTensor'>(tensor([[ 12,  -3, -14,  ..., -12,  28,  45],
        [ 18,  37, -21,  ...,   3, -31,  26],
        [-75,   2,  64,  ...,   6, 104, -18],
        ...,
        [-25,  22, -33,  ..., -11,  89, -19],
        [-14,   8,  33,  ...,  49,  35,  50],
        [-56,  24,  39,  ...,  -8, -12,  23]], device='cuda:0',
       dtype=torch.int8), scale=tensor([[0.0005],
        [0.0004],
        [0.0004],
        ...,
        [0.0004],
        [0.0004],
        [0.0005]], device='cuda:0'), dtype=torch.float32)

In [56]:
compute_model_size(model_one, precision_bytes=1)

'The size of the model is : 0.37749671936035156 GB'

In [57]:
model.device

device(type='cuda', index=0)

In [58]:
model_one.device

device(type='cuda', index=0)